In [0]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Load the data from the workspace folder
df = pd.read_csv('/Workspace/Shared/E-Commerce Sales Insights Dashboard/final_data.csv')

# Display basic info
print(f"Dataset shape: {df.shape}")
print(f"\nColumn names: {df.columns.tolist()}")
print(f"\nFirst few rows:")
display(df.head())

Dataset shape: (10194, 22)

Column names: ['Unnamed: 0', 'Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country/Region', 'City', 'State/Province', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']

First few rows:


Unnamed: 0,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country/Region,City,State/Province,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,US-2023-103800,01/03/2023,01/07/2023,Standard Class,DP-13000,Darren Powers,Consumer,United States,Houston,Texas,77095,Central,OFF-PA-10000174,Office Supplies,Paper,"Message Book, Wirebound, Four 5 1/2"" X 4"" Forms/Pg., 200 Dupl. Sets/Book",16.448,2,0.2,5.5512
1,2,US-2023-112326,01/04/2023,01/08/2023,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,Illinois,60540,Central,OFF-BI-10004094,Office Supplies,Binders,GBC Standard Plastic Binding Systems Combs,3.54,2,0.8,-5.487
2,3,US-2023-112326,01/04/2023,01/08/2023,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,Illinois,60540,Central,OFF-LA-10003223,Office Supplies,Labels,Avery 508,11.784,3,0.2,4.2717
3,4,US-2023-112326,01/04/2023,01/08/2023,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,Illinois,60540,Central,OFF-ST-10002743,Office Supplies,Storage,SAFCO Boltless Steel Shelving,272.736,3,0.2,-64.7748
4,5,US-2023-141817,01/05/2023,01/12/2023,Standard Class,MB-18085,Mick Brown,Consumer,United States,Philadelphia,Pennsylvania,19143,East,OFF-AR-10003478,Office Supplies,Art,"Avery Hi-Liter EverBold Pen Style Fluorescent Highlighters, 4/Pack",19.536,3,0.2,4.884


In [0]:
# Convert date columns to datetime
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date'] = pd.to_datetime(df['Ship Date'])

# Create derived features
df['Profit Margin %'] = (df['Profit'] / df['Sales'] * 100).replace([np.inf, -np.inf], 0).fillna(0)
df['Processing Days'] = (df['Ship Date'] - df['Order Date']).dt.days
df['Order Year'] = df['Order Date'].dt.year
df['Order Month'] = df['Order Date'].dt.month

# Create Order Value Category (Low, Medium, High based on sales quartiles)
df['Order Value Category'] = pd.cut(
    df['Sales'], 
    bins=[0, df['Sales'].quantile(0.33), df['Sales'].quantile(0.66), df['Sales'].max()],
    labels=['Low', 'Medium', 'High'],
    include_lowest=True
)

print("Derived features created successfully!")
print(f"\nDataset now has {df.shape[1]} columns")
print(f"\nNew columns: Profit Margin %, Processing Days, Order Year, Order Month, Order Value Category")
display(df.head())

Derived features created successfully!

Dataset now has 27 columns

New columns: Profit Margin %, Processing Days, Order Year, Order Month, Order Value Category


Unnamed: 0,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country/Region,City,State/Province,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,Profit Margin %,Processing Days,Order Year,Order Month,Order Value Category
0,1,US-2023-103800,2023-01-03T00:00:00.000Z,2023-01-07T00:00:00.000Z,Standard Class,DP-13000,Darren Powers,Consumer,United States,Houston,Texas,77095,Central,OFF-PA-10000174,Office Supplies,Paper,"Message Book, Wirebound, Four 5 1/2"" X 4"" Forms/Pg., 200 Dupl. Sets/Book",16.448,2,0.2,5.5512,33.75,4,2023,1,Low
1,2,US-2023-112326,2023-01-04T00:00:00.000Z,2023-01-08T00:00:00.000Z,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,Illinois,60540,Central,OFF-BI-10004094,Office Supplies,Binders,GBC Standard Plastic Binding Systems Combs,3.54,2,0.8,-5.487,-155.0,4,2023,1,Low
2,3,US-2023-112326,2023-01-04T00:00:00.000Z,2023-01-08T00:00:00.000Z,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,Illinois,60540,Central,OFF-LA-10003223,Office Supplies,Labels,Avery 508,11.784,3,0.2,4.2717,36.25,4,2023,1,Low
3,4,US-2023-112326,2023-01-04T00:00:00.000Z,2023-01-08T00:00:00.000Z,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,Illinois,60540,Central,OFF-ST-10002743,Office Supplies,Storage,SAFCO Boltless Steel Shelving,272.736,3,0.2,-64.7748,-23.75,4,2023,1,High
4,5,US-2023-141817,2023-01-05T00:00:00.000Z,2023-01-12T00:00:00.000Z,Standard Class,MB-18085,Mick Brown,Consumer,United States,Philadelphia,Pennsylvania,19143,East,OFF-AR-10003478,Office Supplies,Art,"Avery Hi-Liter EverBold Pen Style Fluorescent Highlighters, 4/Pack",19.536,3,0.2,4.884,25.0,7,2023,1,Low


In [0]:
# Calculate Core Business KPIs
total_sales = df['Sales'].sum()
total_profit = df['Profit'].sum()
avg_margin = df['Profit Margin %'].mean()
avg_shipping_time = df['Processing Days'].mean()
total_orders = df['Order ID'].nunique()
total_customers = df['Customer ID'].nunique()

print("=== E-COMMERCE SALES INSIGHTS DASHBOARD - KPIs ===")
print(f"\nTotal Sales: ${total_sales:,.2f}")
print(f"Total Profit: ${total_profit:,.2f}")
print(f"Profit Margin: {avg_margin:.2f}%")
print(f"Total Orders: {total_orders:,}")
print(f"Total Customers: {total_customers:,}")
print(f"Avg Processing Time: {avg_shipping_time:.1f} days")
print(f"Avg Order Value: ${total_sales/total_orders:,.2f}")

=== E-COMMERCE SALES INSIGHTS DASHBOARD - KPIs ===

Total Sales: $2,326,534.35
Total Profit: $292,296.81
Profit Margin: 12.22%
Total Orders: 5,111
Total Customers: 804
Avg Processing Time: 4.0 days
Avg Order Value: $455.20


In [0]:
# Monthly Sales and Profit Trend Analysis
monthly_trend = df.groupby(['Order Year', 'Order Month'])[['Sales', 'Profit']].sum().reset_index()
monthly_trend['Date'] = pd.to_datetime(
    monthly_trend['Order Year'].astype(str) + '-' + 
    monthly_trend['Order Month'].astype(str) + '-01'
)

# Create dual-axis chart for Sales and Profit
fig_trend = make_subplots(specs=[[{"secondary_y": True}]])

fig_trend.add_trace(
    go.Scatter(x=monthly_trend['Date'], y=monthly_trend['Sales'], 
               name='Sales', mode='lines+markers', line=dict(color='#2E86AB', width=3)),
    secondary_y=False
)

fig_trend.add_trace(
    go.Scatter(x=monthly_trend['Date'], y=monthly_trend['Profit'], 
               name='Profit', mode='lines+markers', line=dict(color='#06A77D', width=3)),
    secondary_y=True
)

fig_trend.update_layout(
    title='Monthly Sales & Profit Performance Trend',
    template='plotly_white',
    hovermode='x unified',
    height=500
)

fig_trend.update_xaxes(title_text='Timeline')
fig_trend.update_yaxes(title_text='Total Sales ($)', secondary_y=False)
fig_trend.update_yaxes(title_text='Total Profit ($)', secondary_y=True)

fig_trend.show()

In [0]:
# Segmentation Analysis by Order Value Category
segment_analysis = df.groupby('Order Value Category', observed=False).agg(
    Total_Sales=('Sales', 'sum'),
    Order_Count=('Sales', 'count'),
    Avg_Profit=('Profit', 'mean')
).reset_index()

fig_segment = px.bar(
    segment_analysis,
    x='Order Value Category',
    y='Total_Sales',
    color='Order Value Category',
    title='Sales Contribution by Order Value Category',
    labels={'Total_Sales': 'Total Revenue ($)', 'Order Value Category': 'Order Tier'},
    text_auto='.2s',
    color_discrete_map={'Low': '#E63946', 'Medium': '#F77F00', 'High': '#06A77D'}
)

fig_segment.update_layout(template='plotly_white', showlegend=False, height=500)
fig_segment.update_traces(textposition='outside')
fig_segment.show()

In [0]:
# Analyze Processing Time by Ship Mode
shipping_analysis = df.groupby('Ship Mode').agg(
    Avg_Processing_Days=('Processing Days', 'mean'),
    Order_Count=('Order ID', 'count'),
    Total_Sales=('Sales', 'sum')
).reset_index().sort_values(by='Avg_Processing_Days')

fig_shipping = px.bar(
    shipping_analysis,
    x='Ship Mode',
    y='Avg_Processing_Days',
    title='Average Processing Time by Shipping Method',
    labels={'Avg_Processing_Days': 'Avg Days to Ship', 'Ship Mode': 'Shipping Method'},
    color='Avg_Processing_Days',
    color_continuous_scale='RdYlGn_r',
    text_auto='.1f'
)

fig_shipping.update_layout(template='plotly_white', height=500)
fig_shipping.update_traces(textposition='outside')
fig_shipping.show()

In [0]:
# Product Category and Sub-Category Analysis
category_analysis = df.groupby(['Category', 'Sub-Category']).agg(
    Sales=('Sales', 'sum'),
    Profit=('Profit', 'sum'),
    Margin=('Profit Margin %', 'mean')
).reset_index()

fig_category = px.treemap(
    category_analysis,
    path=[px.Constant("All Products"), 'Category', 'Sub-Category'],
    values='Sales',
    color='Profit',
    color_continuous_scale='RdYlGn',
    title='Sales & Profitability by Category and Sub-Category',
    hover_data={'Margin': ':.2f'}
)

fig_category.update_layout(height=600)
fig_category.show()

In [0]:
# Regional and Segment Analysis
region_segment = df.groupby(['Region', 'Segment']).agg(
    Sales=('Sales', 'sum'),
    Profit=('Profit', 'sum'),
    Orders=('Order ID', 'count')
).reset_index()

fig_region = px.bar(
    region_segment,
    x='Region',
    y='Sales',
    color='Segment',
    title='Sales Performance by Region and Customer Segment',
    labels={'Sales': 'Total Sales ($)'},
    barmode='group',
    text_auto='.2s'
)

fig_region.update_layout(template='plotly_white', height=500)
fig_region.show()

In [0]:
# Top Products Analysis
top_products = df.groupby('Product Name').agg(
    Sales=('Sales', 'sum'),
    Profit=('Profit', 'sum'),
    Quantity=('Quantity', 'sum')
).reset_index().sort_values('Sales', ascending=False).head(10)

fig_products = px.bar(
    top_products,
    x='Sales',
    y='Product Name',
    orientation='h',
    title='Top 10 Products by Sales Revenue',
    labels={'Sales': 'Total Sales ($)', 'Product Name': ''},
    color='Profit',
    color_continuous_scale='RdYlGn',
    text_auto='.2s'
)

fig_products.update_layout(template='plotly_white', height=500, yaxis={'categoryorder':'total ascending'})
fig_products.show()